In [1]:
import websocket
import json
import datetime
import threading
import time
import pandas as pd

In [2]:
WEBSOCKET_URL = "wss://ws.bitget.com/v2/ws/public"
INSTRUMENT_IDS = ["SOLUSDT", "BTCUSDT", "ETHUSDT"]
raw_data_df = pd.DataFrame()

In [3]:
def on_open(ws):
    for inst_id in INSTRUMENT_IDS:
        message = {
            "op": "subscribe",
            "args": [{"instType": "SPOT", "channel": "trade", "instId": inst_id}]
        }
        ws.send(json.dumps(message))

def on_message(ws, message_str):  
    global raw_data_df
    try:
        data = json.loads(message_str)
        if "data" in data and data["data"]:
            symbol = data.get("arg", {}).get("instId", "")
            for trade in data["data"]:
                new_row = pd.DataFrame([{
                    'timestamp': datetime.datetime.now(),
                    'symbol': symbol,
                    'data': trade
                }])
                raw_data_df = pd.concat([raw_data_df, new_row], ignore_index=True)
    except:
        pass

def on_error(ws, error):
    pass

def on_close(ws, close_status_code, close_msg):
    pass

In [4]:
a = 60  
b = a * 60  
c = b * 24 
def run_ws_enhanced():
    ws.run_forever(ping_interval=30, ping_timeout=10)


ws = websocket.WebSocketApp(WEBSOCKET_URL,
                          on_open=on_open,
                          on_message=on_message,
                          on_error=on_error,
                          on_close=on_close)




ws_thread = threading.Thread(target=run_ws_enhanced)
ws_thread.daemon = True  
ws_thread.start()

run_duration = a # Chạy trong 1 phút (60 giây)

try:
    time.sleep(run_duration)  
except KeyboardInterrupt:
    print("\n Đã dừng bằng Ctrl+C")  

In [5]:
raw_data_df

,timestamp,symbol,data
0,2025-06-09 09:41:34.923467,SOLUSDT,"{'ts': '1749436888111', 'price': '151.75', 'si..."
1,2025-06-09 09:41:34.923467,SOLUSDT,"{'ts': '1749436888111', 'price': '151.75', 'si..."
2,2025-06-09 09:41:34.923467,SOLUSDT,"{'ts': '1749436888111', 'price': '151.76', 'si..."
3,2025-06-09 09:41:34.932289,SOLUSDT,"{'ts': '1749436888111', 'price': '151.76', 'si..."
4,2025-06-09 09:41:34.932289,SOLUSDT,"{'ts': '1749436888111', 'price': '151.77', 'si..."
...,...,...,...
3851,2025-06-09 09:42:33.519657,ETHUSDT,"{'ts': '1749436953166', 'price': '2492.07', 's..."
3852,2025-06-09 09:42:33.520969,ETHUSDT,"{'ts': '1749436953190', 'price': '2492.07', 's..."
3853,2025-06-09 09:42:33.520969,ETHUSDT,"{'ts': '1749436953210', 'price': '2492.06', 's..."
3854,2025-06-09 09:42:33.552079,BTCUSDT,"{'ts': '1749436953271', 'price': '105544.80', ..."
